In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_error as RMSE
from sklearn.metrics import mean_absolute_percentage_error as MAPE
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as MSE

df = pd.read_csv(r'Dataset\Bearing1_1.csv')
sig = df['PC1'].values


def prepare_data(sig, n, m):
    X, Y = [], []
    # O shift (s) é calculado para alinhar o final de Y com a predição futura
    # Seguindo sua lógica: se n=4, m=3 -> Y começa no índice 2 (hi_3)
    s = n - m + 1 
    
    for i in range(len(sig) - n - 1):
        X.append(np.array([sig[i : i + n]]))
        Y.append(np.array([sig[i + s : i + s + m]]))
        
    #return np.array(X), np.array(Y)
    return X,Y
X,Y = prepare_data(sig,3,2)

In [3]:
import numpy as np

class SimpleRNN:
    def __init__(self, input_dim, hidden_dim, output_dim, lr=0.01):
        np.random.seed(42)
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.lr = lr
        self.k=1
        # Weights
        self.Wxh = np.random.randn(hidden_dim, input_dim) * 0.1
        self.Whh = np.random.randn(hidden_dim, hidden_dim) * 0.1
        self.Why = np.random.randn(output_dim, hidden_dim) * 0.1
        self.bh = np.zeros((hidden_dim, 1))
        self.by = np.zeros((output_dim, 1))

        self.dWxh = np.zeros((hidden_dim, input_dim))
        self.dWhh = np.zeros((hidden_dim, hidden_dim))
        self.dWhy = np.zeros((output_dim, hidden_dim))
        self.dbh = np.zeros((hidden_dim, 1))
        self.dby = np.zeros((output_dim, 1))
        
    def forward(self, inputs):
        """
        inputs: list of arrays, each of shape (input_dim,)
        """
        h = np.zeros((self.hidden_dim, 1))
        self.last_inputs = inputs
        self.last_hs = { -1: h }
        self.last_ys = {}
        self.y_p = None
        for t, x in enumerate(inputs):
            # Ensure x is (input_dim, 1)
            x = x.reshape(-1, 1)
            h = np.tanh(np.dot(self.Wxh, x) + np.dot(self.Whh, h) + self.bh)
            self.last_hs[t] = h
            self.last_ys[t] = np.dot(self.Why, h) + self.by
            self.y_p = np.dot(self.Why, h) + self.by
            self.k +=1
        return [self.last_ys[t] for t in range(len(inputs))]

    def backward(self, targets):
        """
        targets: list of arrays, each of shape (output_dim,)
        """
        dWxh, dWhh, dWhy = np.zeros_like(self.Wxh), np.zeros_like(self.Whh), np.zeros_like(self.Why)
        dbh, dby = np.zeros_like(self.bh), np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_dim, 1))
        #print(self.last_hs)
        for t in reversed(range(len(self.last_inputs))):
            # Targets reshaped to (output_dim, 1)
            target = targets[t].reshape(-1, 1)
            dy = self.last_ys[t] - target
            #print(self.y_p.T,target.T)            
            dWhy += np.dot(dy, self.last_hs[t].T)
            dby += dy
            
            dh = np.dot(self.Why.T, dy) + dh_next
            dh_raw = (1 - self.last_hs[t]**2) * dh
            
            dbh += dh_raw
            dWxh += np.dot(dh_raw, self.last_inputs[t].reshape(1, -1))
            dWhh += np.dot(dh_raw, self.last_hs[t-1].T)
            dh_next = np.dot(self.Whh.T, dh_raw)

        # Gradient Clipping (to prevent exploding gradients)
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)

        for param, dparam in zip([self.Wxh, self.Whh, self.Why, self.bh, self.by], 
                                 [dWxh, dWhh, dWhy, dbh, dby]):
            param -= self.lr * dparam
        
# --- Dynamic Data Preparation ---
ni,nh,no = 3,5,2   # e.g., using current value and its derivative
sig = np.sin(np.linspace(0, 50, 100))
X,Y = prepare_data(sig,ni,no)

# --- Training ---
rnn = SimpleRNN(input_dim=ni, hidden_dim=nh, output_dim=no)

for epoch in range(2):
    loss_acc = 0
    for inputs, targets in zip(X, Y):
        #print('inputs:',inputs,'targets:',targets)
        outputs = rnn.forward(inputs)
        #print(inputs,targets,rnn.y_p)

        # Compute MSE
        loss_acc += np.mean([(out - tar.reshape(-1,1))**2 for out, tar in zip(outputs, targets)])
        rnn.backward(targets)
    
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Avg Loss: {loss_acc/len(X):.6f}")

Epoch 0 | Avg Loss: 0.420584
